# 23 — Demographics vs. Homeless Rate (City/CoC)

Correlations between homeless rate and racial/ethnic composition at CoC level.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_city_data.csv')
df_demo = df.dropna(subset=['homeless_rate_per_10k', 'pct_black']) if 'pct_black' in df.columns else pd.DataFrame()
print(f'CoCs with demographics: {len(df_demo)}')

CoCs with demographics: 0


In [2]:
if len(df_demo) > 5:
    slope, intercept, r, p, se = stats.linregress(df_demo['pct_black'], df_demo['homeless_rate_per_10k'])
    r2 = r ** 2
    x_range = [df_demo['pct_black'].min(), df_demo['pct_black'].max()]
    y_range = [slope * x + intercept for x in x_range]
    fig = px.scatter(
        df_demo, x='pct_black', y='homeless_rate_per_10k', text='city_name',
        color='homeless_rate_per_10k', color_continuous_scale='Reds',
        hover_name='coc_name',
        title=f'% Black Population vs. Homeless Rate (City/CoC)<br><sup>r={r:.3f}, R²={r2:.3f}, p={p:.4f}</sup>',
        labels={'pct_black': '% Black (non-Hispanic)', 'homeless_rate_per_10k': 'Rate per 10k'},
    )
    fig.add_trace(go.Scatter(x=x_range, y=y_range, mode='lines', name='Regression', line=dict(color='darkred', dash='dash')))
    fig.update_traces(textposition='top center', selector=dict(mode='markers+text'))
    fig.show()
else:
    # Show unsheltered % breakdown by state as demographic proxy
    state_avg = df.groupby('state_postal').agg(
        avg_unsheltered_pct=('unsheltered_pct', 'mean'),
        avg_rate=('homeless_rate_per_10k', 'mean'),
        total=('total_homeless', 'sum')
    ).reset_index()
    fig = px.scatter(
        state_avg, x='avg_unsheltered_pct', y='avg_rate', text='state_postal',
        size='total', color='avg_rate', color_continuous_scale='Reds',
        title='Avg Unsheltered % vs. Avg Homeless Rate by State (demographic proxy)',
        labels={'avg_unsheltered_pct': 'Avg Unsheltered %', 'avg_rate': 'Avg Rate per 10k'},
    )
    fig.update_traces(textposition='top center', selector=dict(mode='markers+text'))
    fig.show()

### Political Ideology vs. Homeless Rate (City/CoC)

2020 presidential election (Biden 2-party %) for each CoC primary county.


In [3]:
df_pol = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_city_data.csv')
df_pol = df_pol.dropna(subset=['homeless_rate_per_10k', 'dem_pres_margin_2020'])

slope, intercept, r, p, se = stats.linregress(df_pol['dem_pres_margin_2020'], df_pol['homeless_rate_per_10k'])
r2 = r ** 2
print(f'2020 Dem margin vs homeless rate: r={r:.3f}, R²={r2:.3f}, p={p:.4f}')

x_range = [df_pol['dem_pres_margin_2020'].min(), df_pol['dem_pres_margin_2020'].max()]
y_range = [slope * x + intercept for x in x_range]

fig_pol = px.scatter(
    df_pol, x='dem_pres_margin_2020', y='homeless_rate_per_10k',
    text='city_name',
    color='dem_pres_margin_2020',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    size='total_homeless',
    hover_name='coc_name',
    hover_data={'state_postal': True, 'dem_pres_margin_2020': ':.1f', 'total_homeless': ':,'},
    title=f'2020 Presidential Vote (Dem margin) vs. Homeless Rate (City/CoC)<br>'
          f'<sup>r={r:.3f}, R²={r2:.3f}, p={p:.4f} — bubble size = total homeless</sup>',
    labels={'dem_pres_margin_2020': 'Dem margin (2020 presidential, % points)', 'homeless_rate_per_10k': 'Rate per 10k'},
    size_max=60,
)
fig_pol.add_trace(go.Scatter(x=x_range, y=y_range, mode='lines', name='Regression',
    line=dict(color='black', dash='dash')))
fig_pol.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig_pol.add_vline(x=0, line_dash='dot', line_color='grey', annotation_text='Even split')
fig_pol.show()


2020 Dem margin vs homeless rate: r=0.554, R²=0.307, p=0.0000


In [4]:
# Political lean vs. unsheltered % at city level
slope2, intercept2, r2v, p2, se2 = stats.linregress(df_pol['dem_pres_margin_2020'], df_pol['unsheltered_pct'])
r2_2 = r2v ** 2
print(f'2020 Dem margin vs unsheltered %: r={r2v:.3f}, R²={r2_2:.3f}, p={p2:.4f}')

x_range2 = [df_pol['dem_pres_margin_2020'].min(), df_pol['dem_pres_margin_2020'].max()]
y_range2 = [slope2 * x + intercept2 for x in x_range2]

fig_pol2 = px.scatter(
    df_pol, x='dem_pres_margin_2020', y='unsheltered_pct',
    text='city_name',
    color='dem_pres_margin_2020',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    size='total_homeless',
    hover_name='coc_name',
    title=f'2020 Presidential Vote vs. % Unsheltered Homeless (City/CoC)<br>'
          f'<sup>r={r2v:.3f}, R²={r2_2:.3f}, p={p2:.4f} — bubble size = total homeless</sup>',
    labels={'dem_pres_margin_2020': 'Dem margin (2020 presidential, % points)', 'unsheltered_pct': '% Unsheltered'},
    size_max=60,
)
fig_pol2.add_trace(go.Scatter(x=x_range2, y=y_range2, mode='lines', name='Regression',
    line=dict(color='black', dash='dash')))
fig_pol2.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig_pol2.add_vline(x=0, line_dash='dot', line_color='grey')
fig_pol2.show()


2020 Dem margin vs unsheltered %: r=-0.054, R²=0.003, p=0.5915
